# PS5 — Hardware-OOS Failure Spine + Per-Serial Component Table (notebook-only, v2)

**What this does.** Rebuilds the PS5 failure event as **any hardware OOS 'Set' that lasted ≥ `MIN_OUTAGE_MIN` minutes** — uniformly for TVM / gates / validators — read *directly* from `silver.device_event_enriched` (the 190.9M-row OOS stream), instead of the chargeable-bounded `silver.device_failures` spine. It also builds a per-serial component table with **true per-component OOS counts** and captures **chargeable** events as an analysis column (device + serial level).

**v2 change — the min-outage filter.** Raw hardware OOS counts every transient comms blip (sub-minute self-recovering faults), which made validators look like they "fail" daily → a degenerate ~1-day RUL. This version computes each OOS event's **outage duration from its Set→Clear lifecycle** and drops anything shorter than `MIN_OUTAGE_MIN` (default 5 min), so the survival model sees real outages, not blips.

**Guardrails (read them):**
- **Read-only from silver.** Never writes to any `mars_dev.silver.*` / `mars_dev.gold.*` table. Outputs go to a local folder and are wired into the engine **in memory**.
- **Event = hardware OOS Set (≥ MIN_OUTAGE_MIN)**, NOT chargeable. Chargeable (`FAILURE_LEVEL>0 AND EXCLUDED=0` from `kpi_avail_enriched`) is captured as `total_chargeable_events` / `chargeable_pct` for analysis only.
- **Validators (BMV)** have no chargeable stream → `total_chargeable_events` = 0 by construction.
- **Per-serial is counts + age, not a survival curve** (component-level survival is 100% censored per the catalog).
- **Schema-adaptive + self-reporting.** Column names are resolved at runtime and printed; the min-outage diagnostic shows episodes dropped per type so you can verify the effect.

**Run order:** cells top-to-bottom in the **same kernel as your PS5 engine cell** (so `CONFIG` exists for the wiring cell). Then re-run `main(synthetic=False)`.


In [1]:
# ensure Java for local Spark (run ABOVE Cell 1, then re-run Cell 1)
import os, glob, subprocess

def _find_java_home():
    cands = []
    for base in ("/opt/conda", "/usr/lib/jvm", "/usr/local", "/usr"):
        cands += glob.glob(f"{base}/**/bin/java", recursive=True)
    return os.path.dirname(os.path.dirname(cands[0])) if cands else None

jh = os.environ.get("JAVA_HOME") or _find_java_home()
if not jh:
    print("No JDK found — installing OpenJDK 17 via conda (~1-2 min)…")
    subprocess.run("conda install -y -c conda-forge openjdk=17", shell=True)
    jh = _find_java_home()
assert jh, "Java still not found — use the PS1-kernel fallback below."
os.environ["JAVA_HOME"] = jh
os.environ["PATH"] = f"{jh}/bin:" + os.environ.get("PATH", "")
print("JAVA_HOME =", jh)
subprocess.run(["java", "-version"])

In [2]:
# ── Cell 1 — Spark bootstrap (reuse an active session, else local S3A) ─────────
# Safest: run this in the SAME kernel as your working PS1 validator notebook (its Spark
# session + Java are already configured) — this cell will just reuse it.
import os, sys, subprocess, importlib

def _pip(*pkgs):
    try: ip = get_ipython()
    except NameError: ip = None
    if ip is not None: ip.run_line_magic("pip", "install -q " + " ".join(pkgs))
    else: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    importlib.invalidate_caches()

try:
    import pyspark  # noqa
except ImportError:
    _pip("pyspark==3.5.9")

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

HADOOP_AWS_PACKAGE = "org.apache.hadoop:hadoop-aws:3.3.4"   # matches Spark 3.5 / Hadoop 3.3.x

spark = SparkSession.getActiveSession()
if spark is None:
    os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 22g pyspark-shell")
    spark = (SparkSession.builder.appName("PS5_OOS_Spine_Builder")
             .master("local[*]")
             .config("spark.driver.memory", "22g")
             .config("spark.driver.maxResultSize", "6g")
             .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
             .config("spark.sql.adaptive.enabled", "true")
             .config("spark.jars.packages", HADOOP_AWS_PACKAGE)
             .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                     "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
             .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
             .getOrCreate())
print("Spark:", spark.version, "| master:", spark.sparkContext.master)


In [3]:
# ── Cell 2 — Config (paths, window, scope, min-outage). Mirrors the PS5 engine. ─
GOLD_BUCKET     = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
SCHEME          = "s3a"   # flip to "s3" if your Spark reads use that scheme
def SILVER(t):  return f"{SCHEME}://{GOLD_BUCKET}/chicago/silver/{t}"

RUN_DATE        = "2026-04-11"     # ODS watermark; drop events after this (matches engine RUN_DATE)
TELEMETRY_START = "2024-01-01"     # device_event_enriched window start
SCOPE           = ["TVM", "GATE", "VALIDATOR"]

MIN_OUTAGE_MIN  = 5.0              # drop OOS shorter than this many minutes (transient blips). 0 = keep all.

# operational (non-hardware) events to exclude, per the Data Foundation doc
OPS_EXCLUDE = ["emp logon", "in service", "maintenance mode", "vehicle ignition off", "cmdoosbytables"]

OUT_LOCAL = "PS5_oos_spine_outputs"   # LOCAL scratch (NOT a silver/gold table)
os.makedirs(OUT_LOCAL, exist_ok=True)
print("scope:", SCOPE, "| window:", TELEMETRY_START, "->", RUN_DATE, "| min_outage_min:", MIN_OUTAGE_MIN)


In [4]:
# ── Cell 3 — Helpers + schema probe (verify the adaptive column matches) ─────
def resolve(available, wanted):
    """Case-insensitive: return {role: actual_col_or_None} from candidate lists."""
    low = {c.lower(): c for c in available}
    return {role: next((low[c.lower()] for c in cands if c.lower() in low), None)
            for role, cands in wanted.items()}

def truthy(colname):
    c = F.col(colname)
    return (c.cast("boolean") == F.lit(True)) | \
           (F.lower(F.trim(c.cast("string"))).isin("true", "1", "t", "y", "yes"))

def cols_of(table):
    try:
        return spark.read.parquet(SILVER(table)).limit(1).columns
    except Exception as e:
        print(f"  [probe] {table}: {type(e).__name__} {e}"); return []

for t in ["device_event_enriched", "kpi_avail_enriched", "hw_config_current",
          "dim_failure_level", "maintenance_ledger", "device_mttr"]:
    c = cols_of(t)
    print(f"\n### {t} ({len(c)} cols)\n" + ", ".join(c))


In [12]:
# Cell4 Validator & Gate Exclusions 
MIN_OUTAGE_MIN = 5.0
OUTAGE_CAP_MIN = 43200.0
VAL_EXCL = ["volt drop","devicecommslost","dapcommserror","dap oos","comm loss","host comm lost","cmdoosbytables"]

# --- TVM + VALIDATOR: hardware OOS from device_event_enriched ---
ev = spark.read.parquet(SILVER("device_event_enriched"))
oos = (ev.withColumn("_d", F.to_date(F.col("EVENT_DTM")))
         .filter((F.col("_d") >= F.lit(TELEMETRY_START)) & (F.col("_d") <= F.lit(RUN_DATE)))
         .filter(F.upper(F.col("mars_device_category").cast("string")).isin("TVM","VALIDATOR"))
         .filter(truthy("is_hardware_oos_event")).filter(~truthy("is_commanded_oos_event"))
         .filter(F.col("duration_to_clear_min").isNotNull()))
_v = F.upper(F.col("mars_device_category").cast("string")) == "VALIDATOR"
oos = oos.filter(~(_v & F.lower(F.trim(F.col("EVENT_TYPE_NAME").cast("string"))).isin(*VAL_EXCL)))
oos = oos.withColumn("outage_min", F.least(F.greatest(F.col("duration_to_clear_min").cast("double"), F.lit(0.0)), F.lit(OUTAGE_CAP_MIN))).select(
    F.col("DEVICE_ID").cast("string").alias("DEVICE_ID"), F.col("DEVICE_KEY").cast("string").alias("DEVICE_KEY"),
    F.upper(F.col("mars_device_category").cast("string")).alias("mars_device_category"),
    F.col("_d").alias("failure_date"), F.col("COMPONENT_SERIAL_NBR").cast("string").alias("COMPONENT_SERIAL_NBR"),
    F.col("outage_min").cast("double").alias("outage_min"))

# --- GATE: chargeable from kpi_avail_enriched (FAILURE_LEVEL>0 AND EXCLUDED=0) ---
av = spark.read.parquet(SILVER("kpi_avail_enriched"))
gate = (av.filter(F.upper(F.col("mars_device_category").cast("string")) == "GATE")
          .filter(F.col("FAILURE_LEVEL").cast("double") > 0)
          .filter((F.col("EXCLUDED").cast("double") == 0) | F.col("EXCLUDED").isNull())
          .withColumn("_d", F.to_date(F.col("START_DTM")))
          .filter((F.col("_d") >= F.lit(TELEMETRY_START)) & (F.col("_d") <= F.lit(RUN_DATE)))
          .withColumn("outage_min", F.least(F.greatest(F.col("outage_duration_min").cast("double"), F.lit(0.0)), F.lit(OUTAGE_CAP_MIN))).select(
              F.col("DEVICE_ID").cast("string").alias("DEVICE_ID"), F.col("DEVICE_KEY").cast("string").alias("DEVICE_KEY"),
              F.lit("GATE").alias("mars_device_category"), F.col("_d").alias("failure_date"),
              F.lit(None).cast("string").alias("COMPONENT_SERIAL_NBR"), F.col("outage_min").cast("double").alias("outage_min")))

base_all = oos.unionByName(gate).persist()
# min-outage applies to TVM/VALIDATOR OOS only; gates (chargeable, severe) kept as-is to reproduce the 0.71
base = base_all.filter((F.col("mars_device_category")=="GATE") | (F.col("outage_min") >= F.lit(MIN_OUTAGE_MIN))).persist()

def _epi(d): return d.dropDuplicates(["DEVICE_ID","failure_date"])
_epi(base_all).groupBy("mars_device_category").agg(F.count("*").alias("episodes_raw")).join(
    _epi(base).groupBy("mars_device_category").agg(F.count("*").alias("episodes_kept")),
    "mars_device_category","outer").orderBy("mars_device_category").show()
epi_dev  = _epi(base).persist()
oos_dev  = epi_dev.groupBy("DEVICE_ID","DEVICE_KEY","mars_device_category").agg(F.count("*").alias("total_hardware_oos_events"))
comp_oos = (base.filter(F.col("COMPONENT_SERIAL_NBR").isNotNull())
                .dropDuplicates(["DEVICE_ID","COMPONENT_SERIAL_NBR","failure_date"])
                .groupBy("DEVICE_ID","COMPONENT_SERIAL_NBR").agg(F.count("*").alias("component_oos_events")))
print("kept episodes:", epi_dev.count(), "| devices:", oos_dev.count())

In [13]:
#Cell5 --> chargeable as device-days, comparable to OOS
av = spark.read.parquet(SILVER("kpi_avail_enriched"))
ch = (av.filter(F.col("FAILURE_LEVEL").cast("double") > 0)
        .filter((F.col("EXCLUDED").cast("double") == 0) | F.col("EXCLUDED").isNull())
        .withColumn("_cd", F.to_date(F.col("START_DTM"))).filter(F.col("_cd") <= F.lit(RUN_DATE)))
charge_dev = (ch.dropDuplicates(["DEVICE_KEY","_cd"])          # chargeable-DAYS (matches OOS episode grain)
                .groupBy(F.col("DEVICE_KEY").cast("string").alias("DEVICE_KEY"))
                .agg(F.count("*").alias("total_chargeable_events")))
print("devices with chargeable:", charge_dev.count())

In [14]:
outage_dev = base.groupBy("DEVICE_KEY").agg(F.round(F.expr("percentile_approx(outage_min,0.5)"),1).alias("median_outage_min"))
summary = (oos_dev.join(charge_dev, "DEVICE_KEY", "left")
                  .join(outage_dev, "DEVICE_KEY", "left")
                  .fillna({"total_chargeable_events": 0})
                  .withColumn("chargeable_pct", F.round(100.0*F.col("total_chargeable_events")/
                              F.greatest(F.col("total_hardware_oos_events"), F.lit(1)), 2)).persist())
summary.groupBy("mars_device_category").agg(
    F.countDistinct("DEVICE_ID").alias("devices"),
    F.sum("total_hardware_oos_events").alias("oos_events"),
    F.sum("total_chargeable_events").alias("chargeable_events"),
    F.round(F.avg("median_outage_min"),1).alias("avg_median_outage_min")).show()

In [15]:
# ── Cell 7 — Per-serial component table (hw_config x device summary + age) ────
hw_cfg = spark.read.parquet(SILVER("hw_config_current"))
Rhw = resolve(hw_cfg.columns, {
    "dev":  ["DEVICE_ID", "device_id"],
    "cat":  ["mars_device_category", "device_category"],
    "ser":  ["COMPONENT_SERIAL_NBR", "component_serial_nbr"],
    "desc": ["COMPONENT_DESCRIPTION", "component_description", "COMPONENT_TYPE_NAME"],
    "age":  ["component_age_days", "COMPONENT_AGE_DAYS"],
    "chg":  ["REPORTED_CHANGED_DTM", "reported_changed_dtm"],
})
print("resolved hw_config_current ->", {k: v for k, v in Rhw.items()})
age_col = (F.col(Rhw["age"]).cast("double") if Rhw["age"]
           else F.datediff(F.lit(RUN_DATE), F.to_date(F.col(Rhw["chg"]))).cast("double"))
serial = (hw_cfg.select(
              F.col(Rhw["dev"]).cast("string").alias("DEVICE_ID"),
              F.upper(F.col(Rhw["cat"]).cast("string")).alias("mars_device_category"),
              F.col(Rhw["ser"]).cast("string").alias("COMPONENT_SERIAL_NBR"),
              F.col(Rhw["desc"]).cast("string").alias("COMPONENT_TYPE_NAME"),
              F.greatest(age_col, F.lit(0.0)).alias("component_age_days"))
            .join(summary.select("DEVICE_ID", "total_hardware_oos_events", "total_chargeable_events", "chargeable_pct"),
                  "DEVICE_ID", "left")
            .join(comp_oos, ["DEVICE_ID", "COMPONENT_SERIAL_NBR"], "left")
            .fillna({"total_hardware_oos_events": 0, "total_chargeable_events": 0,
                     "chargeable_pct": 0.0, "component_oos_events": 0})
            .persist())
print("serial rows:", serial.count())
serial.show(6, truncate=False)


In [17]:
# ── Cell 8 — Write compact outputs to LOCAL scratch (NOT silver/gold) ───────
summary_pd = summary.toPandas()
serial_pd  = serial.toPandas()
epi_pd     = epi_dev.toPandas()

summary_pd.to_parquet(f"{OUT_LOCAL}/device_oos_summary.parquet", index=False)
serial_pd.to_parquet(f"{OUT_LOCAL}/device_serial_oos.parquet", index=False)
epi_pd.to_parquet(f"{OUT_LOCAL}/device_oos_episodes.parquet", index=False)
summary_pd.to_csv(f"{OUT_LOCAL}/device_oos_summary.csv", index=False)
serial_pd.to_csv(f"{OUT_LOCAL}/device_serial_oos.csv", index=False)
print("wrote:", OUT_LOCAL, f"({len(summary_pd)} devices, {len(serial_pd)} serials, {len(epi_pd)} episodes)")


## How to run, and what changes

1. Run **Cell 1–8** (Spark). Check three lines: **`outage source`** (which method computed the duration), **`filters applied`** (real filters, not "NONE APPLIED"), and the **episodes_raw → episodes_kept** table — for validators you should see a big drop (the sub-5-min blips), and `chargeable_pct` ≈ 6–8% for TVM/gates, 0 for validators.
2. Run **Cell 9** in the **same kernel as your PS5 engine**, then re-run `results = main(synthetic=False)`.

**What the min-outage filter fixes.** Validators were scoring a ~1-day RUL because every transient comms blip counted as an OOS "failure." Dropping OOS shorter than `MIN_OUTAGE_MIN` (Cell 2, default 5 min) leaves only outages that actually needed attention, so the Weibull median stops collapsing to ~1 day. Tune `MIN_OUTAGE_MIN` up (e.g. 10–15) if validators are still too dense, or to 0 to disable.

**Honest notes.**
- The duration is derived **adaptively**: a duration column if `device_event_enriched` has one, else a clear-timestamp column, else Set→Clear pairing (each Set to its next Clear per device+component). The `outage source` print tells you which path ran — if it says `UNAVAILABLE`, the filter was skipped and you should send me the real clear-time/duration column (or `device_mttr`'s join keys) and I'll wire it exactly.
- Per-serial is **counts + age + risk**, not a survival curve (component-level survival is 100% censored per the catalog). `component_oos_events` is the true per-component OOS count.
- **No silver/gold tables are touched** — outputs live in `PS5_oos_spine_outputs/` and the in-memory `_SYNTH` override.


In [18]:
import pandas as pd
print(pd.read_parquet("PS5_oos_spine_outputs/device_oos_episodes.parquet").groupby("mars_device_category").size())